# Sesgo de género en word embeddings

Basado en [*Man is to Computer Programmer as Woman is to Homemaker? Debiasing Word Embeddings*](https://arxiv.org/abs/1607.06520) (Bolukbasi et al., 2016) y su código oficial: [tolga-b/debiaswe](https://github.com/tolga-b/debiaswe).

## 1. Cargar w2vNEWS (word2vec de Google News)

In [4]:
import json, urllib.request
from types import SimpleNamespace
import numpy as np
import gensim.downloader as api
from gensim.models import KeyedVectors
from sklearn.decomposition import PCA

# w2vNEWS: el word2vec de Google News que usa el paper (300 dimensiones, 3 millones de palabras y frases).
# La primera vez descarga ~1,7 GB en ~/gensim-data. Cargamos solo las palabras más frecuentes (el archivo
# está ordenado por frecuencia): el modelo entero ocupa demasiada memoria y el paper usa las 50.000 primeras.
ruta = api.load("word2vec-google-news-300", return_path=True)
kv = KeyedVectors.load_word2vec_format(ruta, binary=True, limit=500_000)
print(len(kv), "palabras de", kv.vector_size, "dimensiones")

500000 palabras de 300 dimensiones


## 2. El vector de género `g`

Como en la sección 5.1 del paper: diferencias de 10 pares de palabras (*she–he*, *woman–man*...), PCA, y `g` es la primera componente. Los pares y la función `doPCA` son los del repositorio oficial.

Con los vectores normalizados ($\lVert \vec w \rVert = 1$) y los pares $(a_i, b_i)$, $i = 1, \dots, 10$:

1. Centro de cada par: $\mu_i = \dfrac{\vec a_i + \vec b_i}{2}$
2. Se apilan las diferencias al centro, $\vec a_i - \mu_i$ y $\vec b_i - \mu_i$, y se les aplica PCA (equivale a la matriz $C = \sum_i \sum_{w \in \{a_i, b_i\}} (\vec w - \mu_i)^\top (\vec w - \mu_i) / 2$ del paper).
3. $g$ = primera componente principal (vector unitario). Su signo es arbitrario: lo orientamos para que $g \cdot (\vec{she} - \vec{he}) > 0$.
4. Sesgo de una palabra: su proyección sobre $g$, $\;s(w) = \vec w \cdot g = \cos(\vec w, g)$ (> 0: lado *she*; < 0: lado *he*).

*Nota: en la sección 5.1 (Figura 6) el paper normaliza cada diferencia $\vec a_i - \vec b_i$ antes del PCA; su código oficial (y esta celda) sigue la sección 6 sin normalizar. Las dos versiones dan un `g` casi idéntico (coseno 0,99).*

In [5]:
# Los 10 pares de la Figura 5 del paper, tal cual están en su repositorio oficial
URL = "https://raw.githubusercontent.com/tolga-b/debiaswe/master/data/definitional_pairs.json"
pares = json.load(urllib.request.urlopen(URL))
print(pares)

# doPCA copiada del repositorio oficial (debiaswe/we.py): centra cada par en su media y aplica PCA
def doPCA(pairs, embedding, num_components = 10):
    matrix = []
    for a, b in pairs:
        center = (embedding.v(a) + embedding.v(b))/2
        matrix.append(embedding.v(a) - center)
        matrix.append(embedding.v(b) - center)
    matrix = np.array(matrix)
    pca = PCA(n_components = num_components)
    pca.fit(matrix)
    return pca

# doPCA espera un objeto con .v(palabra); le damos los vectores de gensim normalizados (como en el paper)
embedding = SimpleNamespace(v=lambda w: kv.get_vector(w, norm=True))
pca = doPCA(pares, embedding)
g = pca.components_[0]                        # como en debiaswe/debias.py: la primera componente es g
print("Varianza explicada (Figura 6 del paper):", pca.explained_variance_ratio_[:4].round(2))

# El signo de una componente PCA es arbitrario: lo orientamos para que +g apunte hacia "she"
if g @ (embedding.v("she") - embedding.v("he")) < 0:
    g = -g

# Comprobación con la Figura 1 del paper: proyección sobre g (> 0 = lado "she", < 0 = lado "he")
for w in ["homemaker", "nurse", "receptionist", "librarian", "maestro", "captain", "architect", "philosopher"]:
    print(f"{w:14s} {embedding.v(w) @ g:+.3f}")

[['woman', 'man'], ['girl', 'boy'], ['she', 'he'], ['mother', 'father'], ['daughter', 'son'], ['gal', 'guy'], ['female', 'male'], ['her', 'his'], ['herself', 'himself'], ['Mary', 'John']]
Varianza explicada (Figura 6 del paper): [0.61 0.13 0.1  0.05]
homemaker      +0.323
nurse          +0.308
receptionist   +0.280
librarian      +0.277
maestro        -0.244
captain        -0.154
architect      -0.177
philosopher    -0.196


## 3. Jugar con el vector de género

`vecinos(palabra, alpha)` suma `alpha·g` a una palabra y devuelve las palabras más cercanas al resultado (similitud del coseno, con `similar_by_vector` de gensim). `alpha > 0` empuja hacia "she" y `alpha < 0` hacia "he".

$$\text{vecinos}(w, \alpha) = \underset{v}{\text{top-}n} \; \cos(\vec v,\; \vec w + \alpha\, g)$$

In [6]:
def vecinos(palabra, alpha=1.0, topn=10):
    """Palabras más cercanas a palabra + alpha·g  (alpha > 0: hacia "she"; alpha < 0: hacia "he").
    restrict_vocab=50000: solo las 50.000 palabras más frecuentes, como hace el paper."""
    v = embedding.v(palabra) + alpha * g
    return kv.similar_by_vector(v, topn=topn, restrict_vocab=50000)

vecinos("doctor", alpha=0.7)

[('doctor', 0.8233591318130493),
 ('nurse', 0.6871182322502136),
 ('doctors', 0.6210663914680481),
 ('midwife', 0.6157491207122803),
 ('pediatrician', 0.6126316785812378),
 ('physician', 0.6067025661468506),
 ('dermatologist', 0.5820444822311401),
 ('pharmacist', 0.5814012885093689),
 ('dentist', 0.5593457221984863),
 ('oncologist', 0.5500785112380981)]

In [7]:
# La analogía clásica de word2vec con su función de gensim: he : computer_programmer :: she : ?

# kv.most_similar(positive=["she", "computer_programmer"], negative=["he"], topn=10)

kv.most_similar(positive=["woman", "computer_programmer"], negative=["man"], topn=10)

[('homemaker', 0.5627118945121765),
 ('housewife', 0.5105047225952148),
 ('graphic_designer', 0.505180299282074),
 ('schoolteacher', 0.49794942140579224),
 ('businesswoman', 0.493489146232605),
 ('paralegal', 0.49255111813545227),
 ('registered_nurse', 0.4907974600791931),
 ('saleswoman', 0.48816272616386414),
 ('electrical_engineer', 0.4797725975513458),
 ('mechanical_engineer', 0.4755399227142334)]

## Aplicación al proyecto
A continuación vamos a utilizar el vector g para cuantificar si los topics y las profesiones que se muestran tienen sesgo femenino o masculino

In [12]:
topics_female_stereotypes = ["sentimental_life", "family", "motherhood", "children", "home", "house", "kitchen", "cooking", "cleaning", "shopping"]
topics_male_stereotypes = ["sports", "technology", "business", "leadership", "competition", "power", "success", "ambition", "confidence"]

professions_female_stereotypes = ["nurse", "teacher", "secretary", "receptionist", "housekeeper"]
professions_male_stereotypes = ["engineer", "doctor", "lawyer", "manager", "salesman"]

In [13]:
print("\nVecinos de los tópicos femeninos (alpha=0.7):")
for topic in topics_female_stereotypes:
    try:
        print(f"{topic:14s} {embedding.v(topic) @ g:+.3f}")
    except KeyError:
        print(f"{topic:20s} No encontrado")

print("\nVecinos de los tópicos masculinos (alpha=-0.7):")
for topic in topics_male_stereotypes:
    try:
        print(f"{topic:20s} {embedding.v(topic) @ g:+.3f}")
    except KeyError:
        print(f"{topic:20s} No encontrado")

print("\nVecinos de las profesiones femeninas (alpha=0.7):")
for profession in professions_female_stereotypes:
    try:
        print(f"{profession:20s} {embedding.v(profession) @ g:+.3f}")
    except KeyError:
        print(f"{profession:20s} No encontrado")

print("\nVecinos de las profesiones masculinas (alpha=-0.7):")
for profession in professions_male_stereotypes:
    try:
        print(f"{profession:20s} {embedding.v(profession) @ g:+.3f}")
    except KeyError:
        print(f"{profession:20s} No encontrado")



Vecinos de los tópicos femeninos (alpha=0.7):
sentimental_life     No encontrado
family         +0.041
motherhood     +0.401
children       +0.201
home           +0.011
house          +0.078
kitchen        +0.119
cooking        +0.137
cleaning       +0.067
shopping       +0.149

Vecinos de los tópicos masculinos (alpha=-0.7):
sports               -0.089
technology           +0.036
business             -0.017
leadership           -0.080
competition          -0.001
power                -0.078
success              -0.070
ambition             -0.039
confidence           -0.085

Vecinos de las profesiones femeninas (alpha=0.7):
nurse                +0.308
teacher              +0.123
secretary            +0.032
receptionist         +0.280
housekeeper          +0.211

Vecinos de las profesiones masculinas (alpha=-0.7):
engineer             -0.106
doctor               +0.022
lawyer               -0.030
manager              -0.068
salesman             -0.157


## Pendiente: significancia del sesgo

El paper no fija umbrales por palabra (lista extremos, usa $\text{DirectBias}_c$ global y valida con personas). Ideas para más adelante:
- **Por palabra:** percentil de $|s(w)|$ frente al vocabulario (en las 50.000 palabras más frecuentes: p90 ≈ 0,11, p95 ≈ 0,13, p99 ≈ 0,21; el azar da ≈ 0,06) y estabilidad recalculando `g` sin cada par.
- **Por grupos (KPI de profesiones y temas):** test de permutación sobre la diferencia de medias o WEAT (Caliskan et al., 2017), agregando todos los vídeos para tener potencia.